# GPU Experiment 4: Real BM25 RAG & Negative Steering (-v_steer) Evaluation (Device & Dtype Safe)

This notebook implements:
1. **Real BM25 Retrieval-Augmented Generation (RAG)**: Builds a BM25 index over all 14,700 Drug Formulary knowledge contexts, retrieves top-1 passage per query, and measures retrieval recall, generation accuracy, latency, and context token overhead.
2. **Negative Steering (-v_steer)**: Evaluates early-stopping steering with anti-truthful vector direction ($\alpha_0 = -18.0, K=16$) to establish empirical directional specificity.


In [ ]:
!pip install -q rank_bm25 evaluate bert_score bitsandbytes accelerate transformers

import os, sys, json, time, math, torch
import numpy as np
from tqdm import tqdm
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import evaluate

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Count: {torch.cuda.device_count()}")
    print(f"GPU Device 0: {torch.cuda.get_device_name(0)}")


In [ ]:
possible_paths = [
    '/kaggle/input/datasets/anhemgithom/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/vietnamese-medical-halueval-15k/vietnamese_medical_halueval_15k_specialized.json',
    'e:/Paper_Steering_VN_15K/data/vietnamese_medical_halueval_15k_specialized.json',
    './data/vietnamese_medical_halueval_15k_specialized.json',
    './vietnamese_medical_halueval_15k_specialized.json'
]

data_path = None
for p in possible_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError("Dataset file not found!")

with open(data_path, 'r', encoding='utf-8') as f:
    full_dataset = json.load(f)

print(f"Total Dataset Size: {len(full_dataset)} records")
test_data = full_dataset[-500:]
train_data = full_dataset[:-2205]
print(f"Train Pool Size: {len(train_data)} records")
print(f"Unified Test Set Size: {len(test_data)} records")


In [ ]:
print("🔍 Building BM25 Index over Drug Formulary Passages...")
corpus_passages = []
passage_to_id = {}

for idx, item in enumerate(full_dataset):
    ctx = item.get('knowledge_context', item.get('context', '')).strip()
    if ctx and ctx not in passage_to_id:
        passage_to_id[ctx] = len(corpus_passages)
        corpus_passages.append(ctx)

print(f"Total Unique Drug Formulary Passages in Corpus: {len(corpus_passages)}")

def tokenize_text(text):
    return text.lower().split()

corpus_tokens = [tokenize_text(p) for p in corpus_passages]
bm25 = BM25Okapi(corpus_tokens)
print("✅ BM25 Index constructed successfully!")


In [ ]:
model_id = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading {model_id} in 4-bit NF4...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.eval()
bertscore = evaluate.load("bertscore")
print("✅ Model loaded successfully!")


In [ ]:
print("🚀 Running Real BM25 RAG Evaluation...")

def format_bm25_rag_prompt(question, retrieved_context):
    return f"<|im_start|>system\nBạn là một trợ lý y khoa chuyên nghiệp. Hãy trả lời câu hỏi dựa trên tài liệu Dược thư tham khảo dưới đây.<|im_end|>\n<|im_start|>user\n[Tài liệu Dược thư Tham khảo]:\n{retrieved_context}\n\n[Câu hỏi Y khoa]:\n{question}<|im_end|>\n<|im_start|>assistant\n"

rag_gen, rag_refs, rag_hals = [], [], []
latencies, prompt_lens, recall_hits = [], [], []

for item in tqdm(test_data, desc="BM25 RAG Evaluation"):
    q_text = item['question']
    gold_ctx = item.get('knowledge_context', item.get('context', '')).strip()
    
    q_tokens = tokenize_text(q_text)
    top_passages = bm25.get_top_n(q_tokens, corpus_passages, n=1)
    retrieved_ctx = top_passages[0] if top_passages else ""
    
    is_hit = 1 if (gold_ctx and gold_ctx in retrieved_ctx or retrieved_ctx in gold_ctx) else 0
    recall_hits.append(is_hit)
    
    prompt = format_bm25_rag_prompt(q_text, retrieved_ctx)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs.input_ids.shape[1]
    prompt_lens.append(prompt_len)
    
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    latencies.append(time.time() - t0)
    
    gen_text = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)
    rag_gen.append(gen_text)
    rag_refs.append(item.get('right_answer', item.get('positive_answer')))
    rag_hals.append(item['hallucinated_answer'])

bm25_recall = np.mean(recall_hits) * 100
rag_bs_ref = bertscore.compute(predictions=rag_gen, references=rag_refs, model_type="bert-base-multilingual-cased")['f1']
rag_bs_hal = bertscore.compute(predictions=rag_gen, references=rag_hals, model_type="bert-base-multilingual-cased")['f1']
rag_acc = sum(1 for r, h in zip(rag_bs_ref, rag_bs_hal) if r > h) / len(test_data) * 100
rag_bs = np.mean(rag_bs_ref)
mean_latency = np.mean(latencies)
mean_prompt_len = np.mean(prompt_lens)

print("\n=== REAL BM25 RAG EVALUATION RESULTS ===")
print(f"BM25 Retrieval Recall@1: {bm25_recall:.2f}%")
print(f"BM25 RAG RefPref Accuracy: {rag_acc:.2f}%")
print(f"BM25 RAG BERTScore F1: {rag_bs:.4f}")
print(f"BM25 RAG Mean Latency: {mean_latency:.2f}s")
print(f"BM25 RAG Mean Prompt Length: {mean_prompt_len:.1f} tokens")


In [ ]:
print("🚀 Running Negative Steering (-v_steer) Evaluation (Fixed Device & Dtype)...")

def extract_truth_vector(model, tokenizer, train_subset, layer_idx=8):
    pos_acts, neg_acts = [], []
    for item in train_subset[:300]:
        q = item['question']
        pos_ans = item.get('right_answer', item.get('positive_answer'))
        neg_ans = item['hallucinated_answer']
        
        text_pos = f"<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n{pos_ans}"
        text_neg = f"<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n{neg_ans}"
        
        with torch.no_grad():
            inp_pos = tokenizer(text_pos, return_tensors="pt").to(model.device)
            out_pos = model(inp_pos.input_ids, output_hidden_states=True)
            pos_acts.append(out_pos.hidden_states[layer_idx][0, -1, :].detach().cpu())
            
            inp_neg = tokenizer(text_neg, return_tensors="pt").to(model.device)
            out_neg = model(inp_neg.input_ids, output_hidden_states=True)
            neg_acts.append(out_neg.hidden_states[layer_idx][0, -1, :].detach().cpu())
            
    v_diff = torch.stack(pos_acts).mean(dim=0) - torch.stack(neg_acts).mean(dim=0)
    return v_diff / v_diff.norm(p=2)

v_steer = extract_truth_vector(model, tokenizer, train_data, layer_idx=8)

def make_safe_decay_hook_negative(v_vector, alpha_0=-18.0, K=16):
    step_counter = 0
    def hook_fn(module, input_tensor, output_tensor):
        nonlocal step_counter
        step_counter += 1
        if 1 <= step_counter <= K:
            alpha_t = alpha_0 * (1.0 - (step_counter - 1) / K)
            if isinstance(output_tensor, tuple):
                cur_tensor = output_tensor[0]
                v_curr = v_vector.to(device=cur_tensor.device, dtype=cur_tensor.dtype)
                modified = cur_tensor + alpha_t * v_curr
                return (modified,) + output_tensor[1:]
            else:
                v_curr = v_vector.to(device=output_tensor.device, dtype=output_tensor.dtype)
                return output_tensor + alpha_t * v_curr
        return output_tensor
    return hook_fn

def format_standard_prompt(question):
    return f"<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n"

neg_gen, neg_refs, neg_hals = [], [], []
target_layer_module = model.model.layers[8]

for item in tqdm(test_data, desc="Negative Steering Evaluation"):
    q_text = item['question']
    prompt = format_standard_prompt(q_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs.input_ids.shape[1]
    
    hook_handle = target_layer_module.register_forward_hook(make_safe_decay_hook_negative(v_steer, alpha_0=-18.0, K=16))
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    hook_handle.remove()
    
    gen_text = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)
    neg_gen.append(gen_text)
    neg_refs.append(item.get('right_answer', item.get('positive_answer')))
    neg_hals.append(item['hallucinated_answer'])

neg_bs_ref = bertscore.compute(predictions=neg_gen, references=neg_refs, model_type="bert-base-multilingual-cased")['f1']
neg_bs_hal = bertscore.compute(predictions=neg_gen, references=neg_hals, model_type="bert-base-multilingual-cased")['f1']
neg_acc = sum(1 for r, h in zip(neg_bs_ref, neg_bs_hal) if r > h) / len(test_data) * 100
neg_bs = np.mean(neg_bs_ref)

print("\n=== NEGATIVE STEERING (-v_steer) EVALUATION RESULTS ===")
print(f"Negative Steering RefPref Accuracy: {neg_acc:.2f}%")
print(f"Negative Steering BERTScore F1: {neg_bs:.4f}")


In [ ]:
final_summary = {
    "real_bm25_rag": {
        "retrieval_recall_at_1": bm25_recall,
        "accuracy": rag_acc,
        "bertscore_f1": rag_bs,
        "mean_latency_sec": mean_latency,
        "mean_prompt_tokens": mean_prompt_len
    },
    "negative_steering": {
        "alpha_0": -18.0,
        "accuracy": neg_acc,
        "bertscore_f1": neg_bs
    }
}

with open("real_bm25_rag_and_negative_steering_results.json", "w", encoding="utf-8") as f:
    json.dump(final_summary, f, indent=2, ensure_ascii=False)

print("✅ Saved real_bm25_rag_and_negative_steering_results.json successfully!")
print(json.dumps(final_summary, indent=2))
